In [2]:
import pickle
filename = "/home/users/junming.wang/SD-origin/diffusion_head/generated_trajs_0919_all.pkl"
trajs_gen = pickle.load(open(filename, 'rb'))

: 

In [ ]:
trajs_generate = []
for traj in trajs_gen:
    trajs_generate.append(traj.squeeze(0))
print(len(trajs_generate))

In [1]:
import pickle
filename = "/home/users/junming.wang/SD-origin/scripts/features_eval_all_de_with_boxes_with_our_boxes.pkl"
trajs_gt = pickle.load(open(filename, 'rb'))

In [ ]:
import matplotlib.pyplot as plt
import torch

# Two different input tensors
idx = 2

trajectory_1 = trajs_generate[idx]

trajectory_2 = trajs_gt[idx]['ego_trajs'].squeeze(0)

trajectory_3 = trajs_gt[idx]['pred_trajs'].squeeze(0)

# Extract x and y coordinates for both trajectories
x1, y1 = trajectory_1[:, 0].cpu().numpy(), trajectory_1[:, 1].cpu().numpy()
x2, y2 = trajectory_2[:, 0].cpu().numpy(), trajectory_2[:, 1].cpu().numpy()
x3, y3 = trajectory_3[:, 0].cpu().numpy(), trajectory_2[:, 1].cpu().numpy()


# Plotting both trajectories
plt.figure(figsize=(8, 6))
plt.plot(x1, y1, marker='o', linestyle='-', color='b', label='GEN')
plt.plot(x2, y2, marker='o', linestyle='-', color='r', label='GT')
plt.plot(x3, y3, marker='o', linestyle='-', color='g', label='PRED')
plt.axhline(0, color='black', linewidth=1)  # X-axis
plt.axvline(0, color='black', linewidth=1)  # Y-axis
plt.title('Trajectory Plot with (0,0) at X-axis Center')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.legend()
plt.grid(True)
plt.show()

In [7]:
import torch
trajs_gt_eval = []
trajs_pred = []
gt_trajs_mask = []
for traj in trajs_gt:
    # if not traj['gt_ego_fut_masks'].all():
    #     continue
    trajs_gt_eval.append(traj['ego_trajs'].squeeze(0))
    trajs_pred.append(traj['pred_trajs'].squeeze(0))
    gt_trajs_mask.append(traj['gt_ego_fut_masks'].squeeze(0))
device = torch.device('cuda:0')
trajs_gt_eval = torch.stack(trajs_gt_eval,dim=0).to(device)
trajs_pred = torch.stack(trajs_pred,dim=0).to(device)
trajs_generate = torch.stack(trajs_generate,dim=0).to(device)
gt_trajs_mask = torch.stack(gt_trajs_mask,dim=0).to(device)

In [18]:
results_ours = torch.sqrt((((trajs_generate[:, :, :2] - trajs_gt_eval[:, :, :2]) ** 2) * gt_trajs_mask[:,:,:]).sum(dim=-1)) 

In [22]:
results_sparse = torch.sqrt((((trajs_pred[:, :, :2] - trajs_gt_eval[:, :, :2]) ** 2) * gt_trajs_mask[:,:,:]).sum(dim=-1)) 

In [ ]:
#计算L2(0.5-3)
print(torch.mean(results_ours[:,:6]))
print(torch.mean(results_ours[:,:5]))
print(torch.mean(results_ours[:,:4]))
print(torch.mean(results_ours[:,:3]))
print(torch.mean(results_ours[:,:2]))
print(torch.mean(results_ours[:,:1]))

In [ ]:
#计算平均L2
torch.mean(results_ours[:,:6]) + torch.mean(results_ours[:,:5]) +torch.mean(results_ours[:,:4]) + torch.mean(results_ours[:,:3]) + torch.mean(results_ours[:,:2]) + torch.mean(results_ours[:,:1])

In [11]:
import numpy as np
X, Y, Z, W, L, H, SIN_YAW, COS_YAW, VX, VY, VZ = list(range(11))  # undecoded
CNS, YNS = 0, 1  # centerness and yawness indices in quality
YAW = 6 
def box3d_to_corners(box3d):
    if isinstance(box3d, torch.Tensor):
        box3d = box3d.detach().cpu().numpy()
    corners_norm = np.stack(np.unravel_index(np.arange(8), [2] * 3), axis=1)
    corners_norm = corners_norm[[0, 1, 3, 2, 4, 5, 7, 6]]
    # use relative origin [0.5, 0.5, 0]
    corners_norm = corners_norm - np.array([0.5, 0.5, 0.5])
    corners = box3d[:, None, [W, L, H]] * corners_norm.reshape([1, 8, 3])

    # rotate around z axis
    rot_cos = np.cos(box3d[:, YAW])
    rot_sin = np.sin(box3d[:, YAW])
    rot_mat = np.tile(np.eye(3)[None], (box3d.shape[0], 1, 1))
    rot_mat[:, 0, 0] = rot_cos
    rot_mat[:, 0, 1] = -rot_sin
    rot_mat[:, 1, 0] = rot_sin
    rot_mat[:, 1, 1] = rot_cos
    corners = (rot_mat[:, None] @ corners[..., None]).squeeze(axis=-1)
    corners += box3d[:, None, :3]
    return corners

In [6]:

def check_collision(ego_box, boxes):
    '''
        ego_box: tensor with shape [7], [x, y, z, w, l, h, yaw]
        boxes: tensor with shape [N, 7]
    '''
    if  boxes.shape[0] == 0:
        return False

    # follow uniad, add a 0.5m offset
    ego_box[0] += 0.5 * torch.cos(ego_box[6])
    ego_box[1] += 0.5 * torch.sin(ego_box[6])
    ego_corners_box = box3d_to_corners(ego_box.unsqueeze(0))[0, [0, 3, 7, 4], :2]
    corners_box = box3d_to_corners(boxes)[:, [0, 3, 7, 4], :2]
    ego_poly = Polygon([(point[0], point[1]) for point in ego_corners_box])
    for i in range(len(corners_box)):
        box_poly =  Polygon([(point[0], point[1]) for point in corners_box[i]])
        collision = ego_poly.intersects(box_poly)
        if collision:
            return True

    return False

def get_yaw(traj):
    start = traj[0]
    end = traj[-1]
    dist = torch.linalg.norm(end - start, dim=-1)
    if dist < 0.5:
        return traj.new_ones(traj.shape[0]) * np.pi / 2

    zeros = traj.new_zeros((1, 2))
    traj_cat = torch.cat([zeros, traj], dim=0)
    yaw = traj.new_zeros(traj.shape[0]+1)
    yaw[..., 1:-1] = torch.atan2(
        traj_cat[..., 2:, 1] - traj_cat[..., :-2, 1],
        traj_cat[..., 2:, 0] - traj_cat[..., :-2, 0],
    )
    yaw[..., -1] = torch.atan2(
        traj_cat[..., -1, 1] - traj_cat[..., -2, 1],
        traj_cat[..., -1, 0] - traj_cat[..., -2, 0],
    )
    return yaw[1:]

In [7]:
W = 1.85
H = 4.084
def evaluate_single_coll(traj, fut_boxes):
    n_future = traj.shape[0]
    yaw = get_yaw(traj)
    ego_box = traj.new_zeros((n_future, 7))
    ego_box[:, :2] = traj
    ego_box[:, 3:6] = ego_box.new_tensor([H, W, 1.56])
    ego_box[:, 6] = yaw
    collision = torch.zeros(n_future, dtype=torch.bool)

    for t in range(n_future):
        ego_box_t = ego_box[t].clone()
        boxes = fut_boxes[t][0].clone()
        collision[t] = check_collision(ego_box_t, boxes)
    return collision

In [8]:
def evaluate_coll(trajs, gt_trajs, fut_boxes):
    B, n_future, _ = trajs.shape
    trajs = trajs * torch.tensor([-1, 1], device=trajs.device)
    gt_trajs = gt_trajs * torch.tensor([-1, 1], device=gt_trajs.device)

    obj_coll_sum = torch.zeros(n_future, device=trajs.device)
    obj_box_coll_sum = torch.zeros(n_future, device=trajs.device)

    assert B == 1, 'only supprt bs=1'
    for i in range(B):
        gt_box_coll = evaluate_single_coll(gt_trajs[i], fut_boxes)
        box_coll = evaluate_single_coll(trajs[i], fut_boxes)
        box_coll = torch.logical_and(box_coll, torch.logical_not(gt_box_coll))
            
        obj_coll_sum += gt_box_coll.long()
        obj_box_coll_sum += box_coll.long()

    return obj_coll_sum, obj_box_coll_sum

In [ ]:
total = len(trajs_gt)
#get fut_boxes:
from shapely.geometry import Polygon

n_future = 6
obj_coll_sum = torch.zeros(n_future)
obj_box_coll_sum = torch.zeros(n_future)
for i in range(total):
    gt_trajs = trajs_gt[i]['ego_trajs']
    fut_boxes = trajs_gt[i]['fut_boxes']
    trajs = trajs_gt[idx]['pred_trajs']
    obj_coll, obj_box_coll  = evaluate_coll(trajs[:,:,:2], gt_trajs[:,:,:2], fut_boxes[1:])
    #print(obj_coll_sum)
    obj_box_coll_sum += obj_box_coll
    obj_coll_sum += obj_coll
#输出碰撞率结果:
print('obj_coll_sum:', obj_coll_sum)
print('obj_box_coll_sum:', obj_box_coll_sum)